In [1]:
import warnings
warnings.filterwarnings("ignore")


from pathlib import Path

import pandas as pd
import numpy as np

import joblib


from sklearn.model_selection import train_test_split

from sklearn.compose import ColumnTransformer

from sklearn.preprocessing import (
    OneHotEncoder,
    StandardScaler
)

from sklearn.pipeline import Pipeline

from sklearn.impute import SimpleImputer

In [2]:
df = pd.read_csv(
    "../data/processed/churn_clean.csv"
)

df.head()

,gender,SeniorCitizen,Partner,Dependents,tenure,PhoneService,MultipleLines,InternetService,OnlineSecurity,OnlineBackup,DeviceProtection,TechSupport,StreamingTV,StreamingMovies,Contract,PaperlessBilling,PaymentMethod,MonthlyCharges,TotalCharges,Churn
0,Female,0,Yes,No,1,No,No phone service,DSL,No,Yes,No,No,No,No,Month-to-month,Yes,Electronic check,29.85,29.85,0
1,Male,0,No,No,34,Yes,No,DSL,Yes,No,Yes,No,No,No,One year,No,Mailed check,56.95,1889.50,0
2,Male,0,No,No,2,Yes,No,DSL,Yes,Yes,No,No,No,No,Month-to-month,Yes,Mailed check,53.85,108.15,1
3,Male,0,No,No,45,No,No phone service,DSL,Yes,No,Yes,Yes,No,No,One year,No,Bank transfer (automatic),42.30,1840.75,0
4,Female,0,No,No,2,Yes,No,Fiber optic,No,No,No,No,No,No,Month-to-month,Yes,Electronic check,70.70,151.65,1


In [3]:
X = df.drop(
    columns=["Churn"]
)


y = df["Churn"]

In [4]:
X_train, X_test, y_train, y_test = train_test_split(X,y,test_size=0.2,random_state=42,stratify=y)

In [5]:
y_train.value_counts(normalize=True)

Churn
0    0.734222
1    0.265778
Name: proportion, dtype: float64

In [6]:
numeric_features = (
    X_train
    .select_dtypes(
        include=["int64","float64"]
    )
    .columns
)


numeric_features

Index(['SeniorCitizen', 'tenure', 'MonthlyCharges', 'TotalCharges'], dtype='str')

In [7]:
categorical_features = (
    X_train
    .select_dtypes(
        include="object"
    )
    .columns
)


categorical_features

Index(['gender', 'Partner', 'Dependents', 'PhoneService', 'MultipleLines',
       'InternetService', 'OnlineSecurity', 'OnlineBackup', 'DeviceProtection',
       'TechSupport', 'StreamingTV', 'StreamingMovies', 'Contract',
       'PaperlessBilling', 'PaymentMethod'],
      dtype='str')

In [8]:
numeric_transformer = Pipeline(

    steps=[

        (
            "imputer",
            SimpleImputer(
                strategy="median"
            )
        ),

        (
            "scaler",
            StandardScaler()
        )

    ]

)

In [9]:
categorical_transformer = Pipeline(

    steps=[

        (
            "imputer",
            SimpleImputer(
                strategy="most_frequent"
            )
        ),

        (
            "encoder",
            OneHotEncoder(
                handle_unknown="ignore"
            )
        )

    ]

)

In [10]:
preprocessor = ColumnTransformer(

    transformers=[

        (
            "num",
            numeric_transformer,
            numeric_features
        ),

        (
            "cat",
            categorical_transformer,
            categorical_features
        )

    ]

)

In [11]:
X_train_processed = preprocessor.fit_transform(
    X_train
)


X_test_processed = preprocessor.transform(
    X_test
)

In [12]:
X_train_processed.shape

(5625, 45)

In [13]:
Path("../models").mkdir(
    exist_ok=True
)


joblib.dump(
    preprocessor,
    "../models/churn_preprocessor.pkl"
)

['../models/churn_preprocessor.pkl']

In [14]:
!pip install scipy

In [20]:
from scipy import sparse


X_train_sparse = sparse.csr_matrix(X_train_processed)
X_test_sparse = sparse.csr_matrix(X_test_processed)


sparse.save_npz(
    "../data/processed/X_train.npz",
    X_train_sparse
)

sparse.save_npz(
    "../data/processed/X_test.npz",
    X_test_sparse
)


np.save(
    "../data/processed/y_train.npy",
    y_train
)


np.save(
    "../data/processed/y_test.npy",
    y_test
)